# Phase 7: LSTM Baseline Modelling

This notebook trains a 2-layer stacked LSTM on the M5 feature dataset (all 30,490 series),
generates 28-day forecasts for the validation and test sets, and computes baseline metrics
to benchmark against ARIMA (Phase 6) and the later TFT model (Phase 9).

**Key design decisions:**
- `encoder_length = 90` days of history, `decoder_length = 28` days ahead
- Sliding windows with `stride = 28` using a memory-efficient lazy Dataset
- Per-series mean normalisation for the target; StandardScaler for all features
- Huber loss (`delta=1.0`) to handle zero-inflated retail sales
- Gradient clipping at 1.0; Adam + ReduceLROnPlateau; early stopping (patience=10)

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script execution
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

DEVICE = (
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f'Device: {DEVICE}')

PREDS_DIR = Path('../outputs/predictions')
MODELS_DIR = Path('../outputs/models')
FIGS_DIR = Path('../outputs/figures')
for d in [PREDS_DIR, MODELS_DIR, FIGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Device: mps


## 7.1 Load Feature Data

In [2]:
# ── Memory-efficient data loading ─────────────────────────────────────────
# train.parquet has 56 M rows x 67 cols (~34 GB uncompressed).
# We load only required columns and optionally sample series.
# Set N_SERIES=None to use all 30,490 series (~8 GB for these cols).
N_SERIES  = 5000   # set to None for full dataset
LOAD_COLS = ['id', 'date', 'sales', 'cat_id', 'dept_id', 'store_id', 'state_id', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_month_start', 'is_month_end', 'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28']

def _load_split(path, cols=LOAD_COLS, sample_ids=None):
    df = pd.read_parquet(path, columns=cols)
    df['id'] = df['id'].astype('category')
    if sample_ids is not None:
        df = df[df['id'].isin(sample_ids)].copy()
        df['id'] = df['id'].cat.remove_unused_categories()
    return df

# Stratified sample balanced across cat_id x store_id
_meta = pd.read_parquet('../data/features/train.parquet',
                        columns=['id','cat_id','store_id']).drop_duplicates('id')
if N_SERIES and N_SERIES < len(_meta):
    _meta = _meta.groupby(['cat_id','store_id'], group_keys=False).apply(
        lambda g: g.sample(max(1, round(N_SERIES * len(g) / len(_meta))),
                           random_state=42)
    ).head(N_SERIES)
    SAMPLE_IDS = set(_meta['id'].astype(str))
else:
    SAMPLE_IDS = None

train_df = _load_split('../data/features/train.parquet', sample_ids=SAMPLE_IDS)
val_df   = _load_split('../data/features/val.parquet',   sample_ids=SAMPLE_IDS)
test_df  = _load_split('../data/features/test.parquet',  sample_ids=SAMPLE_IDS)

print(f'N_SERIES={N_SERIES}  (None = all 30490)')
print(f'Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}')
print(f'RAM (train): {train_df.memory_usage(deep=True).sum()/1e6:.0f} MB')


N_SERIES=5000  (None = all 30490)
Train: (9285000, 31)  Val: (140000, 31)  Test: (140000, 31)
RAM (train): 892 MB


## 7.2 Feature Selection and Normalisation

We use only features with lag ≥ 28 (no leakage) plus known-future features
(calendar, events, price). Raw sales are excluded from the input — the model
sees historical demand through lag and rolling features only.

In [3]:
CANDIDATE_FEATURE_COLS = [
    # Calendar cyclical
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
    'is_month_start', 'is_month_end',
    # Event / SNAP
    'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting',
    # Price
    'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w',
    # Lags ≥ 28 days (pre-computed, no leakage)
    'lag_28', 'lag_35', 'lag_42', 'lag_56',
    # Rolling stats (computed with 28-day shift, safe)
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28',
    # Static identifiers (integer-encoded)
    'cat_id', 'dept_id', 'store_id', 'state_id',
]

# Keep only columns actually present in the parquet
FEATURE_COLS = [c for c in CANDIDATE_FEATURE_COLS if c in train_df.columns]
N_FEATURES = len(FEATURE_COLS)
ENCODER_LEN = 90
DECODER_LEN = 28

print(f'Feature columns ({N_FEATURES}): {FEATURE_COLS}')

Feature columns (28): ['dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend', 'is_month_start', 'is_month_end', 'is_event', 'is_snap', 'is_cultural', 'is_national', 'is_religious', 'is_sporting', 'sell_price', 'price_vs_category_mean', 'is_price_reduced', 'price_change_1w', 'lag_28', 'lag_35', 'lag_42', 'lag_56', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_28', 'cat_id', 'dept_id', 'store_id', 'state_id']


In [4]:
# Per-series mean-sales scale (used to normalise the TARGET only)
scale_dict = train_df.groupby('id')['sales'].mean().to_dict()
scale_dict = {k: max(float(v), 1.0) for k, v in scale_dict.items()}

# Global StandardScaler fitted on training features
scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS].fillna(0).values.astype(np.float32))

# Apply in-place on copies (originals untouched for later raw re-use)
train_scaled = train_df.copy()
val_scaled   = val_df.copy()
test_scaled  = test_df.copy()

for df_copy in [train_scaled, val_scaled, test_scaled]:
    df_copy[FEATURE_COLS] = scaler.transform(
        df_copy[FEATURE_COLS].fillna(0).values.astype(np.float32)
    ).astype(np.float32)

print('Scaling done.')

Scaling done.


## 7.3 Lazy Sliding-Window Dataset

The dataset stores per-series feature arrays once, then generates windows
on-the-fly in `__getitem__`. This avoids materialising all ~1.9 M windows
simultaneously.

In [5]:
class LazyWindowDataset(Dataset):
    """Memory-efficient sliding-window dataset for LSTM training."""

    def __init__(self, df, feature_cols, scale_dict,
                 encoder_len=90, decoder_len=28, stride=28):
        self.encoder_len = encoder_len
        self.decoder_len = decoder_len
        self.index = []          # (series_id, start_row)
        self.series_data = {}    # series_id -> (features_array, normed_sales_array)

        for sid, grp in df.groupby('id', observed=True):
            grp = grp.sort_values('date').reset_index(drop=True)
            feats = grp[feature_cols].values.astype(np.float32)
            sales = grp['sales'].values.astype(np.float32)
            scale = scale_dict.get(sid, 1.0)
            self.series_data[sid] = (feats, sales / scale)

            n = len(grp)
            max_start = n - encoder_len - decoder_len
            for start in range(0, max_start + 1, stride):
                self.index.append((sid, start))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sid, start = self.index[idx]
        feats, sales_norm = self.series_data[sid]
        end_enc = start + self.encoder_len
        end_dec = end_enc + self.decoder_len
        X = torch.tensor(feats[start:end_enc], dtype=torch.float32)
        y = torch.tensor(sales_norm[end_enc:end_dec], dtype=torch.float32)
        return X, y


STRIDE = 28
full_dataset = LazyWindowDataset(
    train_scaled, FEATURE_COLS, scale_dict,
    encoder_len=ENCODER_LEN, decoder_len=DECODER_LEN, stride=STRIDE
)
print(f'Total training windows: {len(full_dataset):,}')

Total training windows: 315,000


In [6]:
# 90/10 random split for train/val monitoring
val_size   = int(0.10 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_ds, val_ds_monitor = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(f'Train windows: {len(train_ds):,}  |  Val-monitor windows: {len(val_ds_monitor):,}')

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds_monitor, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

Train windows: 283,500  |  Val-monitor windows: 31,500


## 7.4 LSTM Model Architecture

- 2-layer stacked LSTM (`hidden_size=128`, `dropout=0.2`)
- FC decoder: final hidden state → 28 output values
- ~200 K trainable parameters

In [7]:
class LSTMForecasterNotebook(nn.Module):
    """2-layer stacked LSTM encoder → linear decoder for 28-step forecasting."""

    def __init__(self, input_size, hidden_size=128, num_layers=2,
                 dropout=0.2, forecast_horizon=28):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)   # h_n: (layers, batch, hidden)
        return self.fc(h_n[-1])       # (batch, forecast_horizon)


model = LSTMForecasterNotebook(N_FEATURES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

Trainable parameters: 216,604


## 7.5 Training

In [8]:
HIDDEN_SIZE  = 128
MAX_EPOCHS   = 50
PATIENCE     = 10
LR           = 1e-3
WEIGHT_DECAY = 1e-5

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)
criterion = nn.HuberLoss(delta=1.0)

best_val_loss = float('inf')
epochs_no_improve = 0
train_losses_hist = []
val_losses_hist   = []

for epoch in range(MAX_EPOCHS):
    # --- Train ---
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())

    # --- Validate ---
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            val_batch_losses.append(criterion(model(X_batch), y_batch).item())

    tl = float(np.mean(batch_losses))
    vl = float(np.mean(val_batch_losses))
    train_losses_hist.append(tl)
    val_losses_hist.append(vl)
    scheduler.step(vl)

    print(f'Epoch {epoch+1:3d}/{MAX_EPOCHS} | train={tl:.4f} | val={vl:.4f}')

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), MODELS_DIR / 'lstm_best.pt')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f'Early stopping triggered at epoch {epoch+1}.')
            break

print(f'\nBest validation loss: {best_val_loss:.4f}')

Epoch   1/50 | train=0.2708 | val=0.2693
Epoch   2/50 | train=0.2686 | val=0.2688
Epoch   3/50 | train=0.2683 | val=0.2683
Epoch   4/50 | train=0.2676 | val=0.2678
Epoch   5/50 | train=0.2671 | val=0.2678
Epoch   6/50 | train=0.2667 | val=0.2678
Epoch   7/50 | train=0.2663 | val=0.2671
Epoch   8/50 | train=0.2660 | val=0.2665
Epoch   9/50 | train=0.2657 | val=0.2661
Epoch  10/50 | train=0.2654 | val=0.2661
Epoch  11/50 | train=0.2653 | val=0.2663
Epoch  12/50 | train=0.2652 | val=0.2658
Epoch  13/50 | train=0.2650 | val=0.2658
Epoch  14/50 | train=0.2648 | val=0.2658
Epoch  15/50 | train=0.2648 | val=0.2656
Epoch  16/50 | train=0.2648 | val=0.2656
Epoch  17/50 | train=0.2646 | val=0.2655
Epoch  18/50 | train=0.2645 | val=0.2656
Epoch  19/50 | train=0.2645 | val=0.2653
Epoch  20/50 | train=0.2644 | val=0.2654
Epoch  21/50 | train=0.2644 | val=0.2654
Epoch  22/50 | train=0.2642 | val=0.2657
Epoch  23/50 | train=0.2642 | val=0.2654
Epoch  24/50 | train=0.2641 | val=0.2652
Epoch  25/50 | t

In [9]:
# Load best checkpoint
model.load_state_dict(torch.load(MODELS_DIR / 'lstm_best.pt', map_location=DEVICE))
model.eval()
print('Best checkpoint loaded.')

Best checkpoint loaded.


In [10]:
# Plot loss curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses_hist, label='Train Huber Loss', color='steelblue')
ax.plot(val_losses_hist,   label='Val Huber Loss',   color='darkorange')
ax.set_xlabel('Epoch')
ax.set_ylabel('Huber Loss (normalised units)')
ax.set_title('LSTM Training and Validation Loss')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS_DIR / 'lstm_loss_curve.png', dpi=150)
plt.show()
print('Saved lstm_loss_curve.png')

Saved lstm_loss_curve.png


## 7.6 Inference — Validation Set

In [11]:
def generate_predictions(model, history_scaled, target_df, feature_cols,
                         scale_dict, device, encoder_len=90, decoder_len=28):
    """Generate 28-day predictions for every series present in target_df.

    Parameters
    ----------
    history_scaled : DataFrame with feature columns already StandardScaler-transformed.
                     Must contain the encoder_len rows preceding target_df per series.
    target_df      : DataFrame for the forecast period (actual sales available).
    """
    model.eval()
    records = []
    series_ids = target_df['id'].unique()

    for sid in series_ids:
        hist = (
            history_scaled[history_scaled['id'] == sid]
            .sort_values('date')
            .tail(encoder_len)
        )
        if len(hist) < encoder_len:
            continue

        X = hist[feature_cols].values.astype(np.float32)
        X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)

        scale = scale_dict.get(sid, 1.0)
        with torch.no_grad():
            pred_norm = model(X_t).squeeze(0).cpu().numpy()

        pred_sales = np.clip(pred_norm * scale, 0.0, None)

        tgt = target_df[target_df['id'] == sid].sort_values('date').head(decoder_len)
        for d, p, a in zip(tgt['date'].values,
                            pred_sales[:len(tgt)],
                            tgt['sales'].values):
            records.append({'id': sid, 'date': d,
                            'predicted': float(p), 'actual': float(a)})

    return pd.DataFrame(records)


val_preds_df = generate_predictions(
    model, train_scaled, val_df, FEATURE_COLS, scale_dict, DEVICE
)
val_preds_df.to_csv(PREDS_DIR / 'lstm_val_preds.csv', index=False)
print(f'Validation predictions saved: {len(val_preds_df):,} rows')

Validation predictions saved: 140,000 rows


## 7.7 Inference — Test Set

For test inference the encoder window uses the last 90 days of train+val combined.

In [12]:
# Build scaled train+val history for test inference
trainval_raw = pd.concat([train_df, val_df], ignore_index=True)
trainval_scaled = trainval_raw.copy()
trainval_scaled[FEATURE_COLS] = scaler.transform(
    trainval_raw[FEATURE_COLS].fillna(0).values.astype(np.float32)
).astype(np.float32)

test_preds_df = generate_predictions(
    model, trainval_scaled, test_df, FEATURE_COLS, scale_dict, DEVICE
)
test_preds_df.to_csv(PREDS_DIR / 'lstm_test_preds.csv', index=False)
print(f'Test predictions saved: {len(test_preds_df):,} rows')

Test predictions saved: 140,000 rows


## 7.8 Evaluation Metrics

In [13]:
from src.evaluation.metrics import compute_all_metrics

for split_name, preds_df in [('Validation', val_preds_df), ('Test', test_preds_df)]:
    metrics = compute_all_metrics(
        preds_df['actual'].values,
        preds_df['predicted'].values
    )
    print(f'\n=== {split_name} Metrics ===')
    for k, v in metrics.items():
        print(f'  {k}: {v:.4f}')


=== Validation Metrics ===
  MAE: 1.1751
  RMSE: 2.7809
  MAPE: 2705467540.7400
  sMAPE: 153.4235

=== Test Metrics ===
  MAE: 1.1958
  RMSE: 2.8469
  MAPE: 2806086335.2373
  sMAPE: 149.3485


In [14]:
# Metrics by category (using cat_id from val_df / test_df)
print('\n=== Validation Metrics by Category ===')
combined_val = val_preds_df.merge(
    val_df[['id', 'date', 'cat_id']].drop_duplicates(),
    on=['id', 'date'], how='left'
)
for cat, grp in combined_val.groupby('cat_id'):
    m = compute_all_metrics(grp['actual'].values, grp['predicted'].values)
    print(f'  cat={cat} | MAE={m["MAE"]:.3f} | RMSE={m["RMSE"]:.3f} | sMAPE={m["sMAPE"]:.2f}%')


=== Validation Metrics by Category ===
  cat=0 | MAE=1.650 | RMSE=3.667 | sMAPE=140.72%
  cat=1 | MAE=0.586 | RMSE=1.250 | sMAPE=173.03%
  cat=2 | MAE=0.840 | RMSE=1.790 | sMAPE=160.29%
